In [2]:
import pandas as pd
interactions = pd.read_csv(
    "../data/phase2/customer_product_interactions.csv"
)

interactions["last_purchase_date"] = pd.to_datetime(
    interactions["last_purchase_date"],
    dayfirst=True
)

print(interactions.shape)
print(interactions["last_purchase_date"].min())
print(interactions["last_purchase_date"].max())

(119925, 7)
2025-01-01 00:00:00
2026-08-31 00:00:00


In [3]:
cutoff_date = interactions["last_purchase_date"].quantile(0.8)
print("Cutoff date:", cutoff_date)

Cutoff date: 2026-05-03 00:00:00


In [4]:
train_interactions = interactions[
    interactions["last_purchase_date"]<cutoff_date].copy()


test_interactions = interactions[
    interactions["last_purchase_date"] >= cutoff_date
].copy()

print("Training rows:", len(train_interactions))
print("Test rows:", len(test_interactions))

Training rows: 95874
Test rows: 24051


In [5]:
print("Training customers:", train_interactions["customer_id"].nunique())
print("Test customers:", test_interactions["customer_id"].nunique())

Training customers: 3000
Test customers: 2998


In [6]:
test_counts = (
    test_interactions.groupby("customer_id")
    ["product_id"].count()
)
print(test_counts.describe())

count    2998.000000
mean        8.022348
std         2.801014
min         1.000000
25%         6.000000
50%         8.000000
75%        10.000000
max        19.000000
Name: product_id, dtype: float64


In [7]:
ground_truth = (
    test_interactions.groupby("customer_id")
    ["product_id"]
    .apply(set)
    .to_dict()
)

print("Customers with ground truth:", len(ground_truth))
print("Example customer:", list(ground_truth.keys())[0])
print("Future products:", list(ground_truth.values())[0])


Customers with ground truth: 2998
Example customer: C00001
Future products: {'P018153', 'P021815', 'P004291', 'P024325', 'P026988', 'P001745', 'P002519', 'P011166', 'P005206', 'P020091', 'P011860', 'P010525', 'P008293', 'P025706'}


In [8]:
customer_id = "C00001"

train_products = set(
    train_interactions[
        train_interactions["customer_id"] == customer_id
    ]["product_id"]
)

test_products = ground_truth[customer_id]

print("Training products:", len(train_products))
print("Future products:", len(test_products))
print("overlap:", len(train_products & test_products))

Training products: 30
Future products: 14
overlap: 0


In [9]:
train_matrix = train_interactions.pivot_table(
    index = "customer_id",
    columns = "product_id",
    values="total_quantity",
    aggfunc="sum",
    fill_value=0
)
print("Training matrix shape:", train_matrix.shape)

Training matrix shape: (3000, 26599)


In [10]:
from sklearn.metrics.pairwise import cosine_similarity
binary_train = (train_matrix > 0).astype(int)

train_customer_similarity = cosine_similarity(binary_train)

print("Customer Similarity shape:", train_customer_similarity.shape)

Customer Similarity shape: (3000, 3000)


In [11]:
import numpy as np
def train_collaborative_candidates (customer_id, n_similar=20, n_candidates = 50):
    customer_index = train_matrix.index.get_loc(customer_id)

    similarity_scores = train_customer_similarity[customer_index]

    similar_indices = np.argsort(similarity_scores)[::-1]
    candidates = {}
    processed_similar = 0

    for i in similar_indices:
        similar_customer = train_matrix.index[i]
        if similar_customer == customer_id:
            continue
        similarity = similarity_scores[i]
        customer_products = train_matrix.loc[similar_customer]
        purchased_products = customer_products[customer_products>0]
        for product_id, quantity in purchased_products.items():
            if train_matrix.loc[customer_id, product_id] >0 :
                continue
            score = similarity * quantity
            candidates[product_id] = candidates.get(product_id, 0) + score
            processed_similar +=1
            if processed_similar >= n_similar:
                break
    recommendations = (
        pd.DataFrame(
            list(candidates.items()),
            columns=["product_id", "collaborative_score"]
        )
        .sort_values("collaborative_score", ascending=False)
        .head(n_candidates)
    )
    return recommendations

In [12]:
candidates = train_collaborative_candidates(
    "C00001",
    n_similar=20,
    n_candidates=50
)
print(candidates.head(10))

   product_id  collaborative_score
9     P010637             0.149071
11    P011377             0.149071
16    P017394             0.149071
12    P014337             0.149071
18    P022821             0.149071
79    P000249             0.125245
89    P000596             0.120060
58    P000237             0.098374
53    P001154             0.098374
61    P001659             0.096825


In [13]:
content_products = pd.read_csv(
    "../models/content_based/content_products.csv"
)

In [14]:
import joblib
from scipy.sparse import load_npz

tfidf_matrix = load_npz(
    "../models/content_based/tfidf_matrix.npz"
)

tfidf = joblib.load(
    "../models/content_based/tfidf_vectorizer.pkl"
)

In [15]:
customer_id = "C00001"

train_purchased_products = train_interactions[
    train_interactions["customer_id"] == customer_id
]["product_id"].unique()

purchased_indices = content_products[
    content_products["product_id"].isin(train_purchased_products)
].index

candidate_indices = content_products[
    content_products["product_id"].isin(candidates["product_id"])
].index

content_scores =  cosine_similarity(tfidf_matrix[candidate_indices], tfidf_matrix[purchased_indices]).max(axis=1)
candidates["content_score"] = content_scores
print(candidates.head(10))

   product_id  collaborative_score  content_score
9     P010637             0.149071       0.120353
11    P011377             0.149071       0.309696
16    P017394             0.149071       0.170002
12    P014337             0.149071       0.125788
18    P022821             0.149071       0.260012
79    P000249             0.125245       0.081470
89    P000596             0.120060       0.088501
58    P000237             0.098374       0.134636
53    P001154             0.098374       0.098974
61    P001659             0.096825       0.048748


In [16]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

candidates[["collaborative_score", "content_score"]
] = scaler.fit_transform(candidates[["collaborative_score", "content_score"]])
print (candidates.head(10))

   product_id  collaborative_score  content_score
9     P010637             1.000000       0.104234
11    P011377             1.000000       0.313785
16    P017394             1.000000       0.159182
12    P014337             1.000000       0.110249
18    P022821             1.000000       0.258799
79    P000249             0.729900       0.061201
89    P000596             0.671124       0.068982
58    P000237             0.425286       0.120041
53    P001154             0.425286       0.080572
61    P001659             0.407723       0.024986


In [17]:
candidates["hybrid_score"] = (0.6 * candidates["collaborative_score"] + 0.4 * candidates["content_score"])
print(candidates.head(10))

   product_id  collaborative_score  content_score  hybrid_score
9     P010637             1.000000       0.104234      0.641693
11    P011377             1.000000       0.313785      0.725514
16    P017394             1.000000       0.159182      0.663673
12    P014337             1.000000       0.110249      0.644100
18    P022821             1.000000       0.258799      0.703519
79    P000249             0.729900       0.061201      0.462420
89    P000596             0.671124       0.068982      0.430267
58    P000237             0.425286       0.120041      0.303188
53    P001154             0.425286       0.080572      0.287401
61    P001659             0.407723       0.024986      0.254628


In [18]:
top_recommendations = (
    candidates.sort_values("hybrid_score", ascending=False).head(10)
)
print(top_recommendations)

   product_id  collaborative_score  content_score  hybrid_score
11    P011377             1.000000       0.313785      0.725514
18    P022821             1.000000       0.258799      0.703519
16    P017394             1.000000       0.159182      0.663673
12    P014337             1.000000       0.110249      0.644100
9     P010637             1.000000       0.104234      0.641693
78    P000104             0.374951       0.885242      0.579067
35    P000851             0.379912       0.731253      0.520448
7     P009288             0.155051       1.000000      0.493031
79    P000249             0.729900       0.061201      0.462420
89    P000596             0.671124       0.068982      0.430267


In [19]:
recommend_products = set(top_recommendations["product_id"])

future_products = ground_truth["C00001"]

hits = recommend_products & future_products
print("Recommend products:", len(recommend_products))
print("Future products:", len(future_products))
print("Hits:", len(hits))
print("Hit products:", hits)


Recommend products: 10
Future products: 14
Hits: 0
Hit products: set()


In [20]:
precision_at_10 = len(hits)/len(recommend_products)
print("precision@10:", precision_at_10)

precision@10: 0.0


In [21]:
product_to_index = pd.Series(
    content_products.index, 
    index = content_products["product_id"]
).to_dict()
print("Product mappings:", len(product_to_index))

Product mappings: 27555


In [22]:
train_customer_products = (
    train_interactions
    .groupby("customer_id")["product_id"]
    .apply(set)
    .to_dict()

)
print("Customer prepared:", len(train_customer_products))

Customer prepared: 3000


In [23]:
train_customer_tfidf = {}

for customer_id, products in train_customer_products.items():
    indices = [
        product_to_index[p]
        for p in products
        if p in product_to_index
    ]
    if indices:
        train_customer_tfidf[customer_id] = tfidf_matrix[indices]

print("Customers with TF-IDF profiles:", len(train_customer_tfidf))

Customers with TF-IDF profiles: 3000


In [24]:
from tqdm.auto import tqdm
import time

all_eval_recommendations = []
customer_ids = list(ground_truth.keys())
start_time = time.time()
for customer_id in tqdm(
    customer_ids,
    desc = "Evaluating customers",
    unit ="customer"
):
    if customer_id not in train_matrix.index:
        continue
    customer_candidates = train_collaborative_candidates(customer_id, n_similar=20, n_candidates=50)
    candidate_indices = [
        product_to_index[p]
        for p in customer_candidates["product_id"]
        if p in product_to_index
    ]
    candidate_matrix = tfidf_matrix[candidate_indices]
    customer_profile = train_customer_tfidf[customer_id]
    content_scores=cosine_similarity(candidate_matrix,customer_profile).max(axis=1)
    customer_candidates["content_score"] = content_scores
    scaler = MinMaxScaler()
    customer_candidates[
        ["collaborative_score", "content_score"]
    ] = scaler.fit_transform(
        customer_candidates[
            ["collaborative_score", "content_score"]
        ]
    )

    customer_candidates["hybrid_score"] = (
        0.6 * customer_candidates["collaborative_score"] + 0.4 * customer_candidates["content_score"]
    )

    top10 = (
        customer_candidates
        .sort_values("hybrid_score", ascending=False)
        .head(10)
    )

    all_eval_recommendations.append({
        "customer_id": customer_id,
        "recommend_products": set(top10["product_id"])
    })
print ("Customers evaluated:", len(all_eval_recommendations))
print("Total time:",round((time.time() - start_time)/ 60,2),"minutes")

Evaluating customers:   0%|          | 0/2998 [00:00<?, ?customer/s]

Customers evaluated: 2998
Total time: 106.3 minutes


In [26]:
print(all_eval_recommendations[0].keys())

dict_keys(['customer_id', 'recommend_products'])


In [27]:
precision_scores = []
recall_scores = []
hit_rates = []

for result in all_eval_recommendations:
    customer_id = result["customer_id"]
    recommended = result["recommend_products"]
    actual = ground_truth[customer_id]
    hits = recommended & actual
    precision = len(hits) / len(recommended)
    recall = len(hits) / len(actual)
    hit_rate = 1 if len(hits) > 0 else 0
    precision_scores.append(precision)
    recall_scores.append(recall)
    hit_rates.append(hit_rate)

print("precision@10:", sum(precision_scores) / len(precision_scores))
print("Recall@10:", sum(recall_scores) / len(recall_scores))
print("Hit Rate@10:", sum(hit_rates) / len(hit_rates))    

precision@10: 0.00023348899266177454
Recall@10: 0.00026399875048974445
Hit Rate@10: 0.0023348899266177454


In [29]:
print("Total evaluation customers:", len(all_eval_recommendations))
print("\nFirst evaluation result:")
print(all_eval_recommendations[0])
print("\nGround truth for same customer:")
customer_id = all_eval_recommendations[0]["customer_id"]
print(customer_id)
print(ground_truth[customer_id])
print("\nOverlap:")
print(all_eval_recommendations[0]["recommend_products"] & ground_truth[customer_id])

Total evaluation customers: 2998

First evaluation result:
{'customer_id': 'C00001', 'recommend_products': {'P022821', 'P000249', 'P017394', 'P001154', 'P000237', 'P000596', 'P001177', 'P014337', 'P011377', 'P010637'}}

Ground truth for same customer:
C00001
{'P018153', 'P021815', 'P004291', 'P024325', 'P026988', 'P001745', 'P002519', 'P011166', 'P005206', 'P020091', 'P011860', 'P010525', 'P008293', 'P025706'}

Overlap:
set()


In [30]:
overlap_rates = []
for customer_id in ground_truth.keys():
    train_products = train_customer_products.get(customer_id, set())
    future_products = ground_truth[customer_id]
    if len(future_products) > 0:
        overlap_rate = len(train_products & future_products) / len(future_products)
        overlap_rates.append(overlap_rate)

print(sum(overlap_rates) /len(overlap_rates))
print(sum(r > 0 for r in overlap_rates))
print(len(overlap_rates))

0.0
0
2998


In [31]:
transactions = pd.read_csv("../data/phase1/transactions.csv")

repeat_check = (
    transactions.groupby(["customer_id", "product_id"]).size()
)
print (len(repeat_check))
print((repeat_check > 1).sum())
print(repeat_check.max())

119925
75
2


In [32]:
eval_train = []
eval_test = []

for customer_id, group in transactions.groupby("customer_id"):
    group = group.sort_values("purchase_date")

    n_test = max(1, int(len(group) * 0.2))
    eval_test.append(group.tail(n_test))
    eval_train.append(group.iloc[:-n_test])

eval_train = pd.concat(eval_train, ignore_index = True) 
eval_test = pd.concat(eval_test, ignore_index=True)

print(len(eval_train))
print(len(eval_test))
print(eval_train["customer_id"].nunique())

97214
22786
3000


In [33]:
split_overlap = []

for customer_id in eval_test["customer_id"].unique():
    train_products= set(
        eval_train[
            eval_train["customer_id"] == customer_id
        ]['product_id']
    )
    test_products= set(
        eval_test[
            eval_test["customer_id"] == customer_id
        ]['product_id']
    )
    split_overlap.append(len(train_products & test_products))

    print(sum(x > 0 for x in split_overlap))
    print(sum(split_overlap))

0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2


In [34]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from tqdm.auto import tqdm
import numpy as np
import pandas as pd

# -----------------------------------------
# 1. Build TRAIN interaction matrix
# -----------------------------------------

train_matrix = train_interactions.pivot_table(
    index="customer_id",
    columns="product_id",
    values="total_quantity",
    aggfunc="sum",
    fill_value=0
)

# Binary implicit feedback
train_binary = (train_matrix > 0).astype(int)

# Customer similarity using TRAIN data only
train_customer_similarity = cosine_similarity(train_binary)

print("Train interaction matrix:", train_matrix.shape)
print("Customer similarity:", train_customer_similarity.shape)


# -----------------------------------------
# 2. Product ID -> TF-IDF index
# -----------------------------------------

product_to_index = pd.Series(
    content_products.index,
    index=content_products["product_id"]
).to_dict()


# -----------------------------------------
# 3. Generate recommendations
# -----------------------------------------

evaluation_recommendations = []

customer_ids = list(train_matrix.index)

for customer_id in tqdm(
    customer_ids,
    desc="Generating evaluation recommendations"
):

    customer_idx = train_matrix.index.get_loc(customer_id)

    # Similar customers
    similarities = train_customer_similarity[customer_idx].copy()

    # Don't use the customer themselves
    similarities[customer_idx] = -1

    similar_indices = np.argsort(similarities)[::-1][:20]

    candidate_scores = {}

    # Products already purchased in TRAIN
    purchased = set(
        train_matrix.loc[customer_id][
            train_matrix.loc[customer_id] > 0
        ].index
    )

    # Collaborative candidates
    for idx in similar_indices:

        similarity = similarities[idx]

        if similarity <= 0:
            continue

        similar_customer = train_matrix.index[idx]

        products = train_matrix.loc[similar_customer]

        for product_id, quantity in products[products > 0].items():

            if product_id in purchased:
                continue

            score = similarity * quantity

            candidate_scores[product_id] = (
                candidate_scores.get(product_id, 0) + score
            )

    if not candidate_scores:
        continue

    candidates = pd.DataFrame(
        list(candidate_scores.items()),
        columns=["product_id", "collaborative_score"]
    )

    # Keep only top 50 candidates
    candidates = candidates.sort_values(
        "collaborative_score",
        ascending=False
    ).head(50)


    # -----------------------------------------
    # Content scores
    # -----------------------------------------

    purchased_indices = [
        product_to_index[p]
        for p in purchased
        if p in product_to_index
    ]

    candidate_indices = [
        product_to_index[p]
        for p in candidates["product_id"]
        if p in product_to_index
    ]

    if purchased_indices and candidate_indices:

        content_scores = cosine_similarity(
            tfidf_matrix[candidate_indices],
            tfidf_matrix[purchased_indices]
        ).max(axis=1)

        candidates["content_score"] = content_scores

    else:
        candidates["content_score"] = 0


    # -----------------------------------------
    # Normalize + Hybrid score
    # -----------------------------------------

    if len(candidates) > 1:

        scaler = MinMaxScaler()

        candidates[
            ["collaborative_score", "content_score"]
        ] = scaler.fit_transform(
            candidates[
                ["collaborative_score", "content_score"]
            ]
        )

    candidates["hybrid_score"] = (
        0.6 * candidates["collaborative_score"]
        + 0.4 * candidates["content_score"]
    )

    # Top 10
    top10 = (
        candidates
        .sort_values("hybrid_score", ascending=False)
        .head(10)
    )

    evaluation_recommendations.append({
        "customer_id": customer_id,
        "recommended_products": set(top10["product_id"])
    })


print("\nEvaluation completed!")
print(
    "Customers evaluated:",
    len(evaluation_recommendations)
)

Train interaction matrix: (3000, 26599)
Customer similarity: (3000, 3000)


Generating evaluation recommendations:   0%|          | 0/3000 [00:00<?, ?it/s]


Evaluation completed!
Customers evaluated: 3000


In [35]:
precision_scores = []
recall_scores = []
hit_rates = []

for result in evaluation_recommendations:

    customer_id = result["customer_id"]

    recommended = result["recommended_products"]
    actual = ground_truth.get(customer_id, set())

    if len(actual) == 0:
        continue

    hits = recommended & actual

    precision = len(hits) / len(recommended)
    recall = len(hits) / len(actual)
    hit_rate = 1 if len(hits) > 0 else 0

    precision_scores.append(precision)
    recall_scores.append(recall)
    hit_rates.append(hit_rate)

print("Precision@10:", sum(precision_scores) / len(precision_scores))
print("Recall@10:", sum(recall_scores) / len(recall_scores))
print("Hit Rate@10:", sum(hit_rates) / len(hit_rates))

Precision@10: 0.00023348899266177454
Recall@10: 0.00027452163876446733
Hit Rate@10: 0.0023348899266177454


In [36]:
print("Evaluation customers:", len(evaluation_recommendations))

# Check how many recommendations had at least one future purchase
successful_customers = 0

for result in evaluation_recommendations:
    customer_id = result["customer_id"]
    recommended = result["recommended_products"]
    actual = ground_truth.get(customer_id, set())

    if recommended & actual:
        successful_customers += 1

print("Customers with at least 1 correct recommendation:",
      successful_customers)

print("Hit Rate:",
      successful_customers / len(evaluation_recommendations))

Evaluation customers: 3000
Customers with at least 1 correct recommendation: 7
Hit Rate: 0.0023333333333333335
